# 🗜️ 06. 컨텍스트 컴팩션 & 기억 상실 방지 하네스 (5-Stage Compactor & Amnesia Guard)

본 실습 노트북은 프론티어 코딩 에이전트(**Claude Code**, **Hermes**, **LangGraph**)의 핵심 컨텍스트 윈도우 관리 및 긴급 복구 엔진인 **`app/middleware/compaction/`**의 아키텍처를 단계별로 직접 실행하며 학습하는 고급 수업용 교재입니다.

---

### 💡 왜 컨텍스트 컴팩션(Compaction)이 필요한가?

대부분의 단순 에이전트는 대화가 길어지면 다음과 같은 **치명적인 3대 딜레마**에 직면합니다:
1. **컨텍스트 폭발 (Context Overflow)**: `grep`, `read_file`, 웹 검색 등의 도구 출력 몇 번으로 100,000 토큰이 순식간에 차오름.
2. **기억 상실 (Compaction Amnesia)**: 토큰을 아끼려고 전체 대화를 단순히 1개의 요약본(Summary)으로 뭉개면, **방금 수정한 코드 내용이나 진행 중인 계획(Plan)을 에이전트가 잊어버림**.
3. **비상 중단 (413 Payload Too Large)**: 사전 압축을 놓치고 API 호출 시 에러가 터지면 에이전트 루프가 완전히 다운됨.

Claude Code와 최신 하네스 시스템은 이를 해결하기 위해 **점진적 3중 방어선 ➔ 최후의 보루 ➔ 비상 탈출구**로 이어지는 **5단계 다계층 파이프라인**을 구축했습니다.

```mermaid
flowchart LR
    A["입력 메시지"] --> B["1단계: Snip<br>(오래된 턴 1줄 축약)"]
    B --> C["2단계: Micro<br>(5KB+ 대형출력 디스크 스왑)"]
    C --> D["3단계: Collapse<br>(연속 탐색 도구 접기)"]
    D --> E{"토큰 임계치<br>초과?"}
    E -- "No (Auto-Compact 회피 성공!)" --> F["LLM API 호출"]
    E -- "Yes (최후의 보루)" --> G["4단계: Auto-Compact<br>(4-Section 요약 + AmnesiaGuard 복원)"]
    G --> F
    F -- "HTTP 413 에러 발생 시" --> H["5단계: Reactive<br>(턴 단위 20% 긴급 슬라이싱 & 재시도)"]
    H --> F
```

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 핵심 학습 내용 |
|:---:|:---|:---|
| **Step 0** | **환경 세팅 & 격리 샌드박스** | 루트 탐색, `.env` 로드, `nest_asyncio`, 실습 샌드박스(`demo_dir`, `swaps_dir`) 생성 |
| **Part 1** | **[1차 방어선] Snip & Micro Compactor** | 1-1) 턴 기반 오래된 도구 결과 1줄 스니핑<br>1-2) 15,000자 대형 웹 문서 디스크 스왑 & Idempotency(중복 방지)<br>1-3) 인라인 액션 힌트(`read_file` 슬라이스)로 99% 토큰 절감 |
| **Part 2** | **[2차 방어선] Context Collapse (탐색 접기)** | 2-1) [철학 해설] 왜 Collapse가 필요한가? (Auto-Compact 회피 방어선)<br>2-2) 연속 탐색 도구(list/grep/read) 접기 실습<br>2-3) [안전성 검증] `write_file`/`run_command` 등 상태 변경 도구 보존 확인 |
| **Part 3** | **[최후의 보루] Auto Compactor & Amnesia Guard** | 3-1) AmnesiaGuard 도구 인터셉터 (`@wrap_tool_call`)<br>3-2) 토큰 초과 시 4-Section 구조화 LLM 요약<br>3-3) Head(40줄)/Tail(10줄) 스마트 파일 트리밍으로 토큰 폭발 없는 완벽 복원 |
| **Part 4** | **[비상 탈출] Reactive Compactor (413 긴급 복구)** | 4-1) 413 API 에러 시뮬레이션<br>4-2) `HumanMessage` 턴 경계 슬라이싱으로 `AIMessage ↔ ToolMessage` 프로토콜 보존 |
| **Part 5** | **`create_agent`로 프로덕션 에이전트 E2E 실습** | 5-1) 5-Layer Prompt + Compactor + Amnesia 결합 에이전트 가동<br>5-2) 대형 문서 검색 ➔ 스왑 ➔ 부분 조회 멀티턴 시나리오 검증 |
| **Part 6** | **[학습 정리] 컴팩션 계층별 토큰 절감률 & 운영 FAQ** | 단계별 토큰 절감률 비교표 & 프로덕션 최적화 팁 |
| **Part 7** | **🧹 Clean-up & Reset (초기화)** | 샌드박스 임시 디렉토리 및 디스크 스왑 파일 완전 정리 |

---

## 🛠️ Step 0. 환경 세팅 & 격리 샌드박스(Sandbox) 초기화

주피터 환경에서 비동기 루프를 안전하게 실행하기 위해 `nest_asyncio`를 활성화하고, **실습 전용 독립 임시 디렉토리(`demo_dir`)** 및 **디스크 스왑 디렉토리(`demo_swaps_dir`)**를 생성합니다.

In [ ]:
import os
import sys
import json
import shutil
import asyncio
import tempfile
import nest_asyncio
from dotenv import load_dotenv

# 1. 비동기 루프 패치 (Jupyter 환경 필수)
nest_asyncio.apply()

# 2. 프로젝트 루트 동적 탐색 (어느 폴더에서 실행해도 안전하게 연결)
def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), ".."))

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. 환경 변수 로드
dotenv_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path)

# 4. 실습 격리용 샌드박스 디렉토리 생성
demo_dir = tempfile.mkdtemp(prefix="agent_compaction_lab_")
demo_swaps_dir = os.path.join(demo_dir, "swaps")
demo_src_dir = os.path.join(demo_dir, "src")
os.makedirs(demo_swaps_dir, exist_ok=True)
os.makedirs(demo_src_dir, exist_ok=True)

# 5. 실습용 소스코드 파일 생성 (500줄짜리 대형 모듈 - AmnesiaGuard 트리밍 테스트용)
demo_auth_file = os.path.join(demo_src_dir, "auth_service.py")
auth_lines = [
    "# Auth Service v2.0 - Production Security Gateway",
    "import jwt",
    "import hashlib",
    "from datetime import datetime, timedelta",
    "",
    "SECRET_KEY = 'agent-lab-super-secret-key-2026'",
    "",
    "def create_access_token(data: dict, expires_delta: timedelta = None):",
    "    to_encode = data.copy()",
    "    expire = datetime.utcnow() + (expires_delta or timedelta(minutes=15))",
    "    to_encode.update({'exp': expire})",
    "    return jwt.encode(to_encode, SECRET_KEY, algorithm='HS256')",
    "",
    "def verify_access_token(token: str):",
    "    return jwt.decode(token, SECRET_KEY, algorithms=['HS256'])",
]
# 중간에 대량의 헬퍼 함수들 추가 (총 250줄)
for i in range(1, 230):
    auth_lines.append(f"def _internal_helper_policy_{i:03d}(): pass  # validation logic {i}")
auth_lines.extend([
    "",
    "# === RECENT PATCH: Token Blacklist Logic ===",
    "BLACKLISTED_TOKENS = set()",
    "def revoke_token(jti: str):",
    "    BLACKLISTED_TOKENS.add(jti)",
    "def is_token_revoked(jti: str) -> bool:",
    "    return jti in BLACKLISTED_TOKENS",
])

with open(demo_auth_file, "w", encoding="utf-8") as f:
    f.write("\n".join(auth_lines))

print(f"✅ 환경 설정 및 격리 샌드박스 생성 완료!")
print(f"  - Project Root    : {project_root}")
print(f"  - 실습 Sandbox    : {demo_dir}")
print(f"  - 디스크 Swap 경로 : {demo_swaps_dir}")
print(f"  - 실습 소스 파일  : {demo_auth_file} ({len(auth_lines)} 라인, {os.path.getsize(demo_auth_file):,} bytes)")

## 📐 Part 1. [1단계 방어선] Snip & Micro Compactor 실습

### 💡 왜 Snip과 Micro가 분리되어 있고, Snip이 먼저 실행되어야 하는가?
- **SnipCompactor**: **오래된 과거 턴(2턴 이전)**의 도구 결과는 에이전트가 이미 읽고 판단을 끝냈습니다. 따라서 `[Tool result snipped: 500 chars]`처럼 **저렴한 1줄 스텁으로 즉시 압축**합니다.
- **MicroCompactor**: **최근 턴**에서 발생한 **5,000자(5KB) 초과의 대형 출력(웹 문서, 대형 JSON)**을 디스크 파일로 빼두고, 에이전트에게 핀포인트로 읽어올 수 있는 **Actionable Hint**를 제공합니다.
- ⚠️ **중요한 파이프라인 순서 (`Snip ➔ Micro`)**:
  - Snip이 먼저 오래된 도구를 1줄로 줄여야, Micro가 불필요하게 오래된 파일까지 디스크에 쓸데없이 스왑하지 않습니다.
  - 또한 Micro가 스왑한 최신 스텁의 Actionable Hint가 Snip에 의해 지워지는 것을 방지합니다.

### 1-1. Snip Compactor (멀티턴 오래된 도구 결과 점진적 1줄 축약)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from app.middleware.compaction.compactor import SnipCompactor, MicroCompactor

snip = SnipCompactor(age_threshold=2)

# 4턴의 대화 구성 (Turn 1: list_dir, Turn 2: read_file_A, Turn 3: read_file_B, Turn 4: Final query)
multi_turn_messages = [
    SystemMessage(content="[L1-L5 System Stack]"),
    # Turn 1 (오래된 턴 - Snip 대상)
    HumanMessage(content="Turn 1: 프로젝트 파일 목록 보여줘"),
    AIMessage(content="", tool_calls=[{"name": "list_dir", "args": {}, "id": "c1"}]),
    ToolMessage(content="auth_service.py\nserver.py\nconfig.py\n" + ("extra_metadata.txt\n" * 20), tool_call_id="c1", name="list_dir"),
    AIMessage(content="파일 목록 조회를 완료했습니다."),
    # Turn 2 (오래된 턴 - Snip 대상)
    HumanMessage(content="Turn 2: server.py 내용 읽어줘"),
    AIMessage(content="", tool_calls=[{"name": "read_file", "args": {"path": "server.py"}, "id": "c2"}]),
    ToolMessage(content="from fastapi import FastAPI\napp = FastAPI()\n" + ("# routing logic\n" * 20), tool_call_id="c2", name="read_file"),
    AIMessage(content="server.py 파일을 읽었습니다."),
    # Turn 3 (최근 턴 - 원본 온전히 보존!)
    HumanMessage(content="Turn 3: auth_service.py 내용 읽어줘"),
    AIMessage(content="", tool_calls=[{"name": "read_file", "args": {"path": "auth_service.py"}, "id": "c3"}]),
    ToolMessage(content="# Auth Service v2.0\ndef create_access_token(): pass\n" + ("# token details\n" * 20), tool_call_id="c3", name="read_file"),
    AIMessage(content="auth_service.py 파일을 읽었습니다."),
    # Turn 4 (현재 질문)
    HumanMessage(content="Turn 4: auth_service와 server.py를 통합해줘"),
]

snipped_msgs, modified = snip.compact(multi_turn_messages)
print(f"✂️ SnipCompactor 실행 결과 (수정 여부: {modified})")
print("=" * 80)
print(f"- Turn 1 list_dir (과거 턴 축약) : {snipped_msgs[3].content}")
print(f"- Turn 2 server.py (과거 턴 축약) : {snipped_msgs[7].content}")
print(f"- Turn 3 auth_service (최신 턴 유지): {snipped_msgs[11].content[:50]}...")

In [ ]:
for m in snipped_msgs:
    m.pretty_print()

### 1-2. Micro Compactor (대규모 출력 디스크 스왑 & Idempotency)
- 15,000자의 초대형 웹 문서 결과를 디스크(`./artifacts/swaps/swap_*.txt`)로 격리하고 인라인 힌트를 제공합니다.
- 이미 스왑된 스텁은 재처리하지 않는 **Idempotency Guard**가 동작합니다.

In [ ]:
micro = MicroCompactor(max_chars=3000, swap_dir=demo_swaps_dir)

# 15,000자 대규모 웹 문서 생성 (Line 120에 핵심 보안 취약점 정보 포함)
large_web_doc = "<h1>OpenID Connect & OAuth 2.0 Security Best Practices 2026</h1>\n"
for i in range(1, 201):
    if i == 120:
        large_web_doc += f"<section id='vuln'>Section {i}: CRITICAL WARNING - Always validate PKCE code_verifier with SHA-256!</section>\n"
    else:
        large_web_doc += f"<section>Section {i}: Standard token exchange protocols and identity federation metadata details...</section>\n"

print(f"📄 원본 웹 검색 결과 크기: {len(large_web_doc):,} 글자 (약 {len(large_web_doc)//4:,} 토큰)")

web_messages = [
    HumanMessage(content="OAuth2 보안 가이드라인 검색해줘"),
    ToolMessage(content=large_web_doc, tool_call_id="call_oauth_web", name="web_search")
]

micro_compacted, modified = micro.compact(web_messages)
swapped_tool_msg = micro_compacted[1]

print("=" * 80)
print(f"💾 MicroCompactor 적용 후 ToolMessage 스텁:")
print(swapped_tool_msg.content)
print("=" * 80)
print(f"📉 압축 후 메시지 크기: {len(str(swapped_tool_msg.content))} 글자 (절감률: {(1 - len(str(swapped_tool_msg.content))/len(large_web_doc))*100:.1f}%)")

# 2차 실행 (Idempotency 검증): 이미 스왑된 메시지는 다시 파일 생성하지 않고 스킵
re_compacted, re_modified = micro.compact(micro_compacted)
print(f"🔄 Idempotency 재실행 결과 (수정 여부: {re_modified}) ➔ 중복 스왑 파일 생성 없이 안전하게 보존!")

In [ ]:
for m in re_compacted:
    m.pretty_print()

### 1-3. [실전 연동] 스왑된 파일에서 `read_file` 슬라이스로 핀포인트 인출
- 에이전트는 스텁에 포함된 힌트를 보고 15,000자 전체를 다시 읽지 않고, 해당 스왑 파일의 특정 라인(118~123)만 슬라이스 조회합니다.

In [ ]:
import re

# 1. 스텁 힌트에서 swap 파일 경로 추출
match = re.search(r"to disk: ([^\n\]]+)", str(swapped_tool_msg.content))
swap_file_path = match.group(1).strip()
print(f"📍 파싱된 Swap 파일 경로: {swap_file_path}")

# 2. 에이전트의 부분 조회 동작 시뮬레이션 (Line 118 ~ 123 슬라이스)
with open(swap_file_path, "r", encoding="utf-8") as sf:
    lines = sf.readlines()

sliced_lines = lines[118:123]
print("\n🔍 [에이전트가 핀포인트로 읽어온 5줄의 핵심 내용]")
print("─" * 80)
for idx, l in enumerate(sliced_lines, start=119):
    print(f"{idx:03d}: {l.strip()}")
print("─" * 80)
print(f"✨ 15,000자 전체를 컨텍스트에 넣지 않고 {sum(len(l) for l in sliced_lines)}자만 조회하여 목표 달성!")

## 🗂️ Part 2. [2단계 방어선] Context Collapse (탐색 접기 & 변경 도구 보존)

### 💡 Context Collapse는 왜 존재하며 어떻게 동작하는가?
- **목적 (Auto-Compact 회피 방어선)**: 에이전트가 코드베이스를 탐색할 때 `list_dir ➔ grep_search ➔ read_file`을 수없이 반복합니다. 각 출력은 작아서 MicroCompactor가 잡지 못하지만, 합치면 수만 토큰이 됩니다. 이 연속 탐색을 1개의 `[Context Collapsed: N steps]`로 접어서 **대화 전체가 뭉개지는 Auto-compact를 사전에 방어**합니다.

- snip => 오래된 tool 결과, micro => 큰 tool 결과, collapse => 작은 tool 결과지만 자잘한 tool 결과들. 이것이 auto compaction을 유발 가능

- **핵심 안전장치 (`UNSAFE_TO_COLLAPSE_TOOLS`)**:
  - `list_dir`, `grep_search`, `read_file` 등 **순수 조회 도구는 안전하게 접음**.
  - `write_file`, `replace_file_content`, `run_command` 등 **코드를 고치거나 명령어를 실행한 핵심 기록은 절대 접지 않고 100% 보존**.

### 2-1. [비교 실습 1] 순수 탐색 도구 시퀀스는 깔끔하게 1줄 접기

In [ ]:
from app.middleware.compaction.compactor import ContextCollapse

collapse = ContextCollapse(min_consecutive=3, swap_dir=demo_swaps_dir)

# 8단계의 연속 순수 탐색 스텝 시뮬레이션
safe_exploration_messages = [
    SystemMessage(content="[L1-L5 System Stack]"),
    HumanMessage(content="인증 모듈의 모든 의존성을 분석해줘"),
    # 순수 읽기/검색 도구들만 연속 실행
    AIMessage(content="파일 검색 시작", tool_calls=[{"name": "list_dir", "args": {}, "id": "s1"}]),
    ToolMessage(content="src/auth_service.py, src/jwt_util.py, src/db.py", tool_call_id="s1", name="list_dir"),
    AIMessage(content="jwt_util 함수 검색", tool_calls=[{"name": "grep_search", "args": {}, "id": "s2"}]),
    ToolMessage(content="def decode_jwt_token(token): pass", tool_call_id="s2", name="grep_search"),
    AIMessage(content="db 연결 모듈 확인", tool_calls=[{"name": "read_file", "args": {}, "id": "s3"}]),
    ToolMessage(content="class DBConnection: pass", tool_call_id="s3", name="read_file"),
    # 결론 응답
    AIMessage(content="모든 의존성 조사를 마쳤습니다. auth_service는 jwt_util과 db에 의존합니다.")
]

collapsed_msgs, modified = collapse.compact(safe_exploration_messages)
print(f"📦 ContextCollapse 적용 결과 (원본 {len(safe_exploration_messages)}개 메시지 ➔ 압축 후 {len(collapsed_msgs)}개 메시지)")
print("=" * 80)
for idx, m in enumerate(collapsed_msgs):
    role = type(m).__name__
    preview = str(m.content)[:100].replace('\n', ' ')
    print(f"[{idx}] {role:14s}: {preview}...")

# 접힌 시스템 메시지의 tool_call_id 보존 상태 확인
collapsed_sys_msg = collapsed_msgs[2]
print("=" * 80)
print(f"🔑 접힌 SystemMessage의 메타데이터 보존: {collapsed_sys_msg.additional_kwargs.get('collapsed_tool_call_ids')}")

6개의 도구 호출 메시지를 [Context Collapsed: ...]라는 1개의 시스템 메시지로 접어버리면, 원본 tool_call_id들이 프롬프트에서 사라집니다.
하지만 백그라운드 추적기(Tracer)나 감사(Audit) 로깅 시스템에서는 "이 접힌 1줄 뒤에 실제로 어떤 도구 호출들이 실행되었었는가?"를 추적해야 할 때가 있습니다.
따라서 LLM 본문에는 보여주지 않더라도 tool call id를 백업해둔 것 입니다.

### 2-2. [비교 실습 2] `write_file` / `run_command`가 포함된 작업은 접지 않고 보존

In [ ]:
# 파일 수정(write_file) 및 테스트 실행(run_command)이 포함된 시퀀스
modification_messages = [
    SystemMessage(content="[L1-L5 System Stack]"),
    HumanMessage(content="auth_service.py의 만료 로직을 수정하고 테스트를 돌려줘"),
    AIMessage(content="파일 수정 시작", tool_calls=[{"name": "read_file", "args": {}, "id": "m1"}]),
    ToolMessage(content="기존 코드 읽기 완료", tool_call_id="m1", name="read_file"),
    AIMessage(content="새 코드 작성", tool_calls=[{"name": "write_file", "args": {}, "id": "m2"}]),
    ToolMessage(content="File auth_service.py updated successfully.", tool_call_id="m2", name="write_file"),  # ⚠️ UNSAFE!
    AIMessage(content="테스트 실행", tool_calls=[{"name": "run_command", "args": {}, "id": "m3"}]),
    ToolMessage(content="pytest tests/ - 12 passed in 0.4s", tool_call_id="m3", name="run_command"),        # ⚠️ UNSAFE!
    AIMessage(content="수정 및 테스트가 모두 성공했습니다.")
]

mod_result, mod_modified = collapse.compact(modification_messages)
print(f"🛡️ 변경 작업 시퀀스 처리 결과 (수정 여부: {mod_modified})")
print("=" * 80)
print(f"- 원본 메시지 수: {len(modification_messages)}개 == 압축 후 메시지 수: {len(mod_result)}개")
print("✨ write_file과 run_command 기록이 접히지 않고 온전히 보존되어 에이전트가 방금 한 일을 기억합니다!")

## 🧠 Part 3. [최후의 보루] Auto Compactor & Amnesia Guard 실습

### 💡 기억 상실(Amnesia)을 막는 스마트 복원 원리
1. **AutoCompactor**: Snip/Micro/Collapse로도 토큰이 8,000을 넘으면, 전체 대화를 LLM을 통해 **4개 섹션(목표, 결정, 코드 변경점, 다음 작업)**으로 요약합니다.
2. **AmnesiaGuard**: 요약이 끝난 직후, 최근 작업한 파일의 **Head(상위 40줄) + Tail(하위 10줄)**과 활성 계획(Active Plan)을 SystemMessage로 자동 주입합니다.
3. **토큰 폭발 방지**: 대형 파일 전체를 통째로 부어 토큰이 다시 폭발하는 것을 막고, 생략된 부분은 `read_file`로 확인하라는 액션 힌트를 제공합니다.

In [ ]:
from app.middleware.compaction.amnesia_guard import AmnesiaGuardMiddleware, create_amnesia_guard_middleware
from app.middleware.compaction.compactor import AutoCompactor
from app.utils import init_chat_model

# 1. AmnesiaGuard 초기화 및 250줄짜리 실습 소스코드 + 계획 등록
guard = AmnesiaGuardMiddleware(max_restore_files=3, max_file_chars=3000)
guard.track_file_access(demo_auth_file)
guard.set_active_plan([
    "1. JWT 발급 함수 구현 [완료]",
    "2. JWT 만료 검증 로직 추가 [진행 중]",
    "3. SQLite 토큰 블랙리스트 연동 [대기]"
])

# 2. AutoCompactor 초기화 (threshold=300 tokens로 설정하여 실습)
try:
    llm = init_chat_model(model="gemini-3.7-flash", temperature=0.0)
except Exception:
    # Fallback to lightweight mock
    from unittest.mock import MagicMock
    llm = MagicMock()
    llm.get_num_tokens_from_messages = MagicMock(return_value=1500)
    llm.invoke = MagicMock(return_value=AIMessage(content="""1. Primary Goal: Implement JWT token verification
2. Key Decisions: Use HS256 algorithm with 15-min expiration
3. Code Modifications: Created auth_service.py and helper policies
4. Current Task: Verify access token expiration logic"""))

auto_compactor = AutoCompactor(llm=llm, threshold_tokens=300, amnesia_guard=guard)

# 3. 긴 대화 히스토리 구성
long_history = [
    SystemMessage(content="[L1-L5 Claude Code Prompt Stack]"),
    HumanMessage(content="FastAPI에서 JWT 인증 구조를 설계해줘. 상세히 부탁해."),
    AIMessage(content="FastAPI 보안 아키텍처:\n" + ("- OAuth2PasswordBearer를 사용하여 의존성 주입을 구성합니다.\n" * 15)),
    HumanMessage(content="토큰 만료 시간 설정 방법도 알려줘."),
    AIMessage(content="토큰 만료 시간은 timedelta를 사용하여 다음과 같이 구현합니다:\n" + ("- ACCESS_TOKEN_EXPIRE_MINUTES = 30\n" * 15)),
    HumanMessage(content="좋아, 방금 작성한 auth_service.py의 만료 시간 검증 코드를 수정해보자.")
]

compacted_auto, modified = auto_compactor.compact_if_needed(long_history)

print(f"⚡ AutoCompactor & AmnesiaGuard 실행 완료 (수정 여부: {modified})")
print("=" * 80)
for idx, m in enumerate(compacted_auto):
    role = type(m).__name__
    preview = str(m.content)[:100].replace('\n', ' ')
    print(f"[{idx}] {role:14s}: {preview}...")

print("=" * 80)
print("📋 [AmnesiaGuard가 스마트 Head/Tail 트리밍으로 복원 주입한 실물]")
print(compacted_auto[2].content)

## 🚨 Part 4. [비상 탈출] Reactive Compactor (413 에러 긴급 복구)

사전 압축에도 불구하고 LLM API 호출 시 413(Payload Too Large) 에러가 발생했을 때, 에이전트 루프가 중단되지 않고 **`HumanMessage` 턴 경계 단위 20% Silent Slicing + 자동 재시도**를 수행합니다.

```
[0] HumanMessage (Turn 1)
[1] AIMessage (tool_call: c1)   ← 인덱스로 자르면 AI만 잘려나가고 Tool은 남아서 고아 메시지 발생!
[2] ToolMessage (result: c1)    ← 턴 경계 단위로 자르면 [0]~[2]가 한 번에 잘려 API 에러 방지!
```

In [ ]:
from app.middleware.compaction.compactor import create_compactor_middleware

# 1. 413 에러를 모의 발생시키는 핸들러 구성
call_count = 0

class MockRequest:
    def __init__(self, messages):
        self.messages = list(messages)
    def override(self, **kwargs):
        return MockRequest(kwargs.get("messages", self.messages))

def flaky_api_handler(request):
    global call_count
    call_count += 1
    if call_count == 1:
        print("💥 [API Server] 413 Client Error: Request Entity Too Large (context_length_exceeded)")
        raise Exception("413 Client Error: Request Entity Too Large (context_length_exceeded)")
    print("🎉 [API Server Retry Success] 턴 경계 20% 슬라이싱 후 정상 재시도 성공!")
    return AIMessage(content="긴급 복구 후 정상 생성된 LLM 응답입니다.")

# 2. 통합 컴팩션 미들웨어 생성
compactor_mw = create_compactor_middleware(
    llm=llm,
    threshold_tokens=50000,
    amnesia_guard=guard,
    swap_dir=demo_swaps_dir
)

# 3. 5턴의 멀티턴 대화 구성
five_turn_messages = [SystemMessage(content="[L1-L5 System Stack]")]
for i in range(5):
    five_turn_messages.append(HumanMessage(content=f"Turn {i+1} User Question"))
    five_turn_messages.append(AIMessage(content=f"Turn {i+1} AI Answer"))

req = MockRequest(five_turn_messages)
result = compactor_mw.wrap_model_call(req, flaky_api_handler)

print("=" * 80)
print(f"최종 응답: {result.content}")
print(f"총 호출 시도 횟수: {call_count}회 (1회 실패 ➔ 턴 경계 슬라이싱 ➔ 1회 자동 재시도 성공)")

## 🤖 Part 5. `create_agent`로 프로덕션 에이전트 E2E 실습

실제 LangChain `create_agent`에 **PromptAssembler + CompactorMiddleware + AmnesiaGuard**를 모두 결합하여 멀티턴 실전 에이전트를 가동합니다.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from app.middleware.prompt import PromptAssembler, create_prompt_assembler_middleware

# 1. 실습용 대형 도구 정의
@tool
def search_security_docs(query: str) -> str:
    """Search security guidelines (returns large payload > 6,000 chars)."""
    return f"[DOCS: {query}]\n" + ("- RFC 6749 OAuth 2.0 Security Specification Details\n" * 120)

# 2. 프롬프트 어셈블러 및 컴팩터 미들웨어 구성
prompt_assembler = PromptAssembler(system_rules="You are a professional security and software engineering AI assistant.")
prompt_mw = create_prompt_assembler_middleware(prompt_assembler, merge_system=False)
compactor_mw = create_compactor_middleware(llm=llm, threshold_tokens=8000, amnesia_guard=guard, swap_dir=demo_swaps_dir)
amnesia_tool_mw = create_amnesia_guard_middleware(guard)

# 3. 프로덕션 에이전트 조립
agent = create_agent(
    model=llm,
    tools=[search_security_docs],
    middleware=[prompt_mw, compactor_mw, amnesia_tool_mw],
    checkpointer=MemorySaver()
)

print("✨ 5-Layer Prompt + Compactor + AmnesiaGuard 결합 에이전트 생성 완료!")

# 4. 멀티턴 실행 검증
config = {"configurable": {"thread_id": "compaction_demo_thread_01"}}
try:
    response = await agent.ainvoke({
        "messages": [HumanMessage(content="OAuth2 보안 스펙을 search_security_docs로 검색해줘")]
    }, config=config)
    print("=" * 80)
    print("🤖 에이전트 응답:")
    print(str(response["messages"][-1].content)[:300] + "...")
except Exception as e:
    print(f"(API 호출 완료/시뮬레이션): {e}")

# 5. 스왑 디렉토리에 swap 파일이 정상 생성되었는지 확인
created_swaps = os.listdir(demo_swaps_dir)
print("=" * 80)
print(f"💾 디스크 스왑 파일 생성 확인: {created_swaps}")

## 📊 Part 6. [학습 정리] 컴팩션 계층별 토큰 절감률 및 운영 가이드

### 1. 계층별 토큰 절감률 비교

| 컴팩션 계층 | 주요 대상 | 압축 방식 | 토큰 절감률 | 맥락 보존율 |
|:---|:---|:---|:---:|:---:|
| **1. Snip Compactor** | 2턴 이전의 오래된 도구 결과 | 1줄 텍스트 스텁 대체 | **~90%** | 높음 (과거 판단 유지) |
| **2. Micro Compactor** | 5,000자 초과 대형 웹/문서 결과 | 디스크 스왑 + 액션 힌트 | **~99%** | 완벽 (필요 시 부분 인출) |
| **3. Context Collapse** | 연속된 읽기/검색 탐색 도구 | Collapsed SystemMessage 접기 | **~95%** | 완벽 (변경 도구는 100% 보존) |
| **4. Auto Compactor** | 8,000 토큰 초과 누적 대화 | 4-Section LLM 요약 + Amnesia 복원 | **~98%** | 완벽 (최신 파일/계획 재주입) |
| **5. Reactive Compactor** | 413 Context Overflow 에러 | 턴 단위 20% 긴급 슬라이싱 | **~25%** | 높음 (AI-Tool 쌍 보존) |

---

### 2. 프로덕션 운영 FAQ

Q. **Context Collapse에서 어떤 도구가 접히고 어떤 도구가 보존되나요?**
- `list_dir`, `grep_search`, `read_file`, `glob` 등 순수 조회 도구는 안전하게 접힙니다.
- `write_file`, `replace_file_content`, `run_command`, `git_commit` 등 상태를 바꾸는 도구는 절대 접히지 않고 온전히 보존됩니다.

Q. **AmnesiaGuard가 복원하는 파일이 너무 크면 토큰이 다시 폭발하지 않나요?**
- Head(40줄) + Tail(10줄) 스마트 트리밍과 `max_file_chars=3000` 상한선이 적용되어 있어, 대형 파일이라도 최대 ~750 토큰 이내로 안전하게 복원됩니다.

## 🧹 Part 7. Clean-up & Reset (샌드박스 정리)

실습에 사용된 임시 디렉토리 및 스왑 파일들을 안전하게 삭제하고 정리합니다.

In [ ]:
if os.path.exists(demo_dir):
    shutil.rmtree(demo_dir, ignore_errors=True)
    print(f"🧹 실습 임시 디렉토리 정리 완료: {demo_dir}")
else:
    print("이미 정리되었습니다.")